# 🎓 Student Performance, GPA & Dropout Risk Modeling
## Multi-Task Deep Learning with TensorFlow / Keras

This notebook documents the research, data synthesis, neural network architecture, and evaluation for predicting:
1. **Future GPA** (Continuous Regression)
2. **Dropout Risk** (Binary Classification)
3. **Academic Performance Tier** (Multi-class Classification)


### 1. Academic Dataset Generation & Missing Value Handling


In [4]:
import numpy as np
import pandas as pd

# ==========================================
# Academic Dataset
# ==========================================

np.random.seed(42)

n_samples = 5000

StudentID = np.arange(1, n_samples + 1)

# -------------------------------
# Personal Information
# -------------------------------

Age = np.random.randint(18, 30, n_samples)

Gender = np.random.randint(0, 2, n_samples)

Department = np.random.choice(
    [0, 1, 2, 3, 4],
    size=n_samples,
    p=[0.22, 0.20, 0.18, 0.20, 0.20]
)

Semester = np.random.randint(1, 9, n_samples)

# -------------------------------
# Family Background
# -------------------------------

FamilyIncome = np.clip(
    np.random.normal(60, 18, n_samples),
    20,
    120
)

ParentEducation = np.random.choice(
    [0, 1, 2, 3],
    size=n_samples,
    p=[0.18, 0.32, 0.34, 0.16]
)

Scholarship = (
    (FamilyIncome < 50).astype(int)
)

# -------------------------------
# Academic Ability
# -------------------------------

HighSchoolGPA = np.clip(
    np.random.normal(
        15.5 + ParentEducation * 0.4,
        1.8,
        n_samples
    ),
    8,
    20
)

StudyHoursPerWeek = np.clip(
    np.random.normal(
        18 + (HighSchoolGPA - 15),
        5,
        n_samples
    ),
    5,
    40
)

SleepHours = np.clip(
    np.random.normal(
        7.2 - (StudyHoursPerWeek - 18) * 0.04,
        0.7,
        n_samples
    ),
    4,
    9
)

InternetUsage = np.clip(
    np.random.normal(
        4.5 - (StudyHoursPerWeek - 18) * 0.08,
        1,
        n_samples
    ),
    1,
    8
)

LibraryVisits = np.clip(
    np.random.normal(
        StudyHoursPerWeek / 3,
        2,
        n_samples
    ),
    0,
    20
).round()

ClassParticipation = np.clip(
    np.random.normal(
        60 + StudyHoursPerWeek,
        10,
        n_samples
    ),
    20,
    100
)

AttendanceRate = np.clip(
    np.random.normal(
        72 +
        (StudyHoursPerWeek * 0.8) -
        (InternetUsage * 2),
        6,
        n_samples
    ),
    40,
    100
)

AssignmentAverage = np.clip(
    np.random.normal(
        45 +
        HighSchoolGPA * 2.2 +
        StudyHoursPerWeek * 0.6,
        6,
        n_samples
    ),
    40,
    100
)

QuizAverage = np.clip(
    np.random.normal(
        AssignmentAverage * 0.90,
        5,
        n_samples
    ),
    35,
    100
)

MidtermScore = np.clip(
    np.random.normal(
        (
            AssignmentAverage +
            QuizAverage +
            AttendanceRate
        ) / 3,
        5,
        n_samples
    ),
    35,
    100
)

CurrentGPA = np.clip(
    np.random.normal(
        (
            HighSchoolGPA * 0.45 +
            (AssignmentAverage / 5) * 0.20 +
            (QuizAverage / 5) * 0.20 +
            (MidtermScore / 5) * 0.15
        ),
        0.5,
        n_samples
    ),
    8,
    20
)

PreviousFailures = np.clip(
    (
        (
            16 - CurrentGPA
        ) / 2 +
        np.random.normal(
            0,
            0.5,  
            n_samples
        )
    ).round(),
    0,
    6
).astype(int)

# -------------------------------
# Create DataFrame
# -------------------------------

academic_df = pd.DataFrame({

    "StudentID": StudentID,
    "Age": Age,
    "Gender": Gender,
    "Department": Department,
    "Semester": Semester,

    "FamilyIncome": FamilyIncome.round(2),
    "ParentEducation": ParentEducation,
    "Scholarship": Scholarship,

    "HighSchoolGPA": HighSchoolGPA.round(2),
    "CurrentGPA": CurrentGPA.round(2),

    "StudyHoursPerWeek": StudyHoursPerWeek.round(1),
    "SleepHours": SleepHours.round(1),
    "InternetUsage": InternetUsage.round(1),
    "LibraryVisits": LibraryVisits.astype(int),

    "AttendanceRate": AttendanceRate.round(1),
    "AssignmentAverage": AssignmentAverage.round(1),
    "QuizAverage": QuizAverage.round(1),
    "MidtermScore": MidtermScore.round(1),

    "ClassParticipation": ClassParticipation.round(1),
    "PreviousFailures": PreviousFailures

})


In [5]:
for col in ["HighSchoolGPA", "QuizAverage", "MidtermScore","FamilyIncome"]:
    academic_df.loc[academic_df.sample(frac=0.2, random_state=42).index, col] = np.nan


In [6]:
print(academic_df.head())

print(academic_df.describe())

print(academic_df.info())

print(academic_df.isnull().sum())

   StudentID  Age  Gender  Department  Semester  FamilyIncome  \
0          1   24       0           1         2         33.23   
1          2   21       1           0         5         61.57   
2          3   28       1           1         6         82.45   
3          4   25       0           4         8         62.44   
4          5   22       0           1         8         87.74   

   ParentEducation  Scholarship  HighSchoolGPA  CurrentGPA  StudyHoursPerWeek  \
0                2            1          14.02       15.46               22.2   
1                2            0          16.75       17.69               22.3   
2                1            0          15.85       16.90               22.0   
3                3            0          17.00       17.99               23.3   
4                3            0          17.29       16.51                7.8   

   SleepHours  InternetUsage  LibraryVisits  AttendanceRate  \
0         7.3            3.2              3            74.6

In [7]:
academic_df["HighSchoolGPA"] = academic_df["HighSchoolGPA"].fillna(academic_df["HighSchoolGPA"].mean())
academic_df["QuizAverage"] = academic_df["QuizAverage"].fillna(academic_df["QuizAverage"].mean())
academic_df["MidtermScore"] = academic_df["MidtermScore"].fillna(academic_df["MidtermScore"].mean())
academic_df["FamilyIncome"] = academic_df["FamilyIncome"].fillna(academic_df["FamilyIncome"].mean())


In [8]:
academic_df.isnull().sum()

StudentID             0
Age                   0
Gender                0
Department            0
Semester              0
FamilyIncome          0
ParentEducation       0
Scholarship           0
HighSchoolGPA         0
CurrentGPA            0
StudyHoursPerWeek     0
SleepHours            0
InternetUsage         0
LibraryVisits         0
AttendanceRate        0
AssignmentAverage     0
QuizAverage           0
MidtermScore          0
ClassParticipation    0
PreviousFailures      0
dtype: int64

### 2. Online Learning (LMS) Dataset Generation & Imputation


In [9]:

# ==========================================
# Online Learning Dataset
# ==========================================

StudentID = np.arange(1, n_samples + 1)

# ----------------------------------------
# Student Engagement Level (Hidden Variable)
# ----------------------------------------

Engagement = np.clip(
    np.random.normal(70, 15, n_samples),
    20,
    100
)

# ----------------------------------------
# Login Behavior
# ----------------------------------------

LoginFrequency = np.clip(
    np.random.normal(
        Engagement / 3,
        3,
        n_samples
    ),
    2,
    40
).round()

TotalLoginTime = np.clip(
    np.random.normal(
        LoginFrequency * 2.5,
        10,
        n_samples
    ),
    20,
    180
)

# ----------------------------------------
# Learning Activities
# ----------------------------------------

VideosWatched = np.clip(
    np.random.normal(
        LoginFrequency * 3,
        8,
        n_samples
    ),
    5,
    120
).round()

CompletionRate = np.clip(
    np.random.normal(
        Engagement,
        8,
        n_samples
    ),
    20,
    100
)

DownloadedResources = np.clip(
    np.random.normal(
        VideosWatched * 0.55,
        4,
        n_samples
    ),
    0,
    60
).round()

PracticeTestsTaken = np.clip(
    np.random.normal(
        VideosWatched / 5,
        2,
        n_samples
    ),
    0,
    25
).round()

AveragePracticeScore = np.clip(
    np.random.normal(
        CompletionRate * 0.90,
        6,
        n_samples
    ),
    30,
    100
)

# ----------------------------------------
# Forum Activities
# ----------------------------------------

ForumPosts = np.clip(
    np.random.normal(
        Engagement / 10,
        2,
        n_samples
    ),
    0,
    20
).round()

QuestionsAsked = np.clip(
    np.random.normal(
        ForumPosts * 0.8,
        2,
        n_samples
    ),
    0,
    20
).round()

QuestionsAnswered = np.clip(
    np.random.normal(
        ForumPosts * 0.9,
        2,
        n_samples
    ),
    0,
    20
).round()

# ----------------------------------------
# Assignment Behavior
# ----------------------------------------

AssignmentSubmissionDelay = np.clip(
    np.random.normal(
        15 - Engagement / 8,
        2,
        n_samples
    ),
    0,
    15
)

ConsecutiveInactiveDays = np.clip(
    np.random.normal(
        12 - Engagement / 10,
        2,
        n_samples
    ),
    0,
    15
).round()

CourseCompletionTime = np.clip(
    np.random.normal(
        18 - Engagement / 8,
        3,
        n_samples
    ),
    5,
    20
)

# ----------------------------------------
# Device Usage
# ----------------------------------------

MobileUsage = np.clip(
    np.random.normal(
        65,
        18,
        n_samples
    ),
    10,
    100
)

DesktopUsage = np.clip(
    100 - MobileUsage +
    np.random.normal(0, 5, n_samples),
    0,
    100
)

NightStudyPercentage = np.clip(
    np.random.normal(
        35,
        15,
        n_samples
    ),
    0,
    100
)

WeekendStudyPercentage = np.clip(
    np.random.normal(
        30,
        12,
        n_samples
    ),
    0,
    100
)

AIUsageFrequency = np.clip(
    np.random.normal(
        Engagement / 10,
        2,
        n_samples
    ),
    0,
    15
).round()

# ----------------------------------------
# Course Certificate
# ----------------------------------------

CertificateEarned = (
    (
        (CompletionRate > 80) &
        (AveragePracticeScore > 70)
    )
).astype(int)

# ----------------------------------------
# Create DataFrame
# ----------------------------------------

online_df = pd.DataFrame({

    "StudentID": StudentID,

    "LoginFrequency": LoginFrequency.astype(int),
    "TotalLoginTime": TotalLoginTime.round(1),

    "VideosWatched": VideosWatched.astype(int),
    "CompletionRate": CompletionRate.round(1),

    "ForumPosts": ForumPosts.astype(int),
    "QuestionsAsked": QuestionsAsked.astype(int),
    "QuestionsAnswered": QuestionsAnswered.astype(int),

    "AssignmentSubmissionDelay": AssignmentSubmissionDelay.round(1),

    "DownloadedResources": DownloadedResources.astype(int),

    "PracticeTestsTaken": PracticeTestsTaken.astype(int),
    "AveragePracticeScore": AveragePracticeScore.round(1),

    "ConsecutiveInactiveDays": ConsecutiveInactiveDays.astype(int),

    "MobileUsage": MobileUsage.round(1),
    "DesktopUsage": DesktopUsage.round(1),

    "NightStudyPercentage": NightStudyPercentage.round(1),
    "WeekendStudyPercentage": WeekendStudyPercentage.round(1),

    "AIUsageFrequency": AIUsageFrequency.astype(int),

    "CourseCompletionTime": CourseCompletionTime.round(1),

    "CertificateEarned": CertificateEarned

})


In [10]:
for col in ["LoginFrequency", "ConsecutiveInactiveDays", "DownloadedResources","AIUsageFrequency"]:
    online_df.loc[online_df.sample(frac=0.2, random_state=42).index, col] = np.nan


In [11]:
print(online_df.head())

print(online_df.describe())

print(online_df.info())

print(online_df.isnull().sum())

   StudentID  LoginFrequency  TotalLoginTime  VideosWatched  CompletionRate  \
0          1            31.0            83.7             83            80.2   
1          2             8.0            28.0             19            53.9   
2          3            22.0            68.7             68            70.0   
3          4            25.0            68.7             66            65.0   
4          5            34.0            83.1             85           100.0   

   ForumPosts  QuestionsAsked  QuestionsAnswered  AssignmentSubmissionDelay  \
0           6               4                  5                        8.1   
1           5               4                  2                        8.3   
2           9               8                 10                        3.6   
3           5               1                  4                        6.1   
4          11              10                 10                        3.9   

   DownloadedResources  PracticeTestsTaken  Averag

In [12]:
online_df["LoginFrequency"] = online_df["LoginFrequency"].fillna(online_df["LoginFrequency"].mean())
online_df["ConsecutiveInactiveDays"] = online_df["ConsecutiveInactiveDays"].fillna(online_df["ConsecutiveInactiveDays"].mean())
online_df["DownloadedResources"] = online_df["DownloadedResources"].fillna(online_df["DownloadedResources"].mean())
online_df["AIUsageFrequency"] = online_df["AIUsageFrequency"].fillna(online_df["AIUsageFrequency"].mean())

In [13]:
online_df.isnull().sum()

StudentID                    0
LoginFrequency               0
TotalLoginTime               0
VideosWatched                0
CompletionRate               0
ForumPosts                   0
QuestionsAsked               0
QuestionsAnswered            0
AssignmentSubmissionDelay    0
DownloadedResources          0
PracticeTestsTaken           0
AveragePracticeScore         0
ConsecutiveInactiveDays      0
MobileUsage                  0
DesktopUsage                 0
NightStudyPercentage         0
WeekendStudyPercentage       0
AIUsageFrequency             0
CourseCompletionTime         0
CertificateEarned            0
dtype: int64

### 3. Feature Synthesis & Target Construction (Multi-Task Labels)


In [14]:
df = pd.concat([
    online_df,
    academic_df.drop(columns="StudentID")

], axis=1)

df.head()

,StudentID,LoginFrequency,TotalLoginTime,VideosWatched,CompletionRate,ForumPosts,QuestionsAsked,QuestionsAnswered,AssignmentSubmissionDelay,DownloadedResources,...,StudyHoursPerWeek,SleepHours,InternetUsage,LibraryVisits,AttendanceRate,AssignmentAverage,QuizAverage,MidtermScore,ClassParticipation,PreviousFailures
0,1,31.0,83.7,83,80.2,6,4,5,8.1,53.0,...,22.2,7.3,3.2,3,74.6,88.1,65.8,70.3,94.0,1
1,2,8.0,28.0,19,53.9,5,4,2,8.3,12.0,...,22.3,6.2,4.2,8,77.6,92.1,83.2,83.8,99.4,0
2,3,22.0,68.7,68,70.0,9,8,10,3.6,37.0,...,22.0,7.2,3.9,5,87.7,90.4,90.4,89.2,81.2,0
3,4,25.0,68.7,66,65.0,5,1,4,6.1,29.0,...,23.3,7.6,2.4,9,85.6,93.1,85.2,85.4,95.8,0
4,5,34.0,83.1,85,100.0,11,10,10,3.9,43.0,...,7.8,7.1,5.1,0,68.0,84.7,76.5,78.1,42.3,0


In [15]:
noise = np.random.normal(0, 0.22, len(df))

df["FutureGPA"] = (

    0.40 * df["CurrentGPA"]

    + 0.15 * (df["AssignmentAverage"] / 5)

    + 0.10 * (df["QuizAverage"] / 5)

    + 0.08 * (df["MidtermScore"] / 5)

    + 0.08 * (df["AttendanceRate"] / 5)

    + 0.05 * (df["StudyHoursPerWeek"] / 2)

    + 0.05 * (df["AveragePracticeScore"] / 5)

    + 0.04 * (df["CompletionRate"] / 5)

    + 0.03 * (df["LoginFrequency"] / 2)

    + 0.02 * df["LibraryVisits"]

    - 0.18 * df["PreviousFailures"]

    - 0.10 * df["ConsecutiveInactiveDays"]

    - 0.06 * df["AssignmentSubmissionDelay"]

    - 0.04 * df["InternetUsage"]

    + noise

)

df["FutureGPA"] = np.clip(df["FutureGPA"], 0, 20).round(2)


In [16]:
noise = np.random.normal(0, 0.2, len(df))

risk_score = (

      0.18 * df["PreviousFailures"]

    + 0.12 * df["ConsecutiveInactiveDays"]

    + 0.10 * df["AssignmentSubmissionDelay"]

    + 0.08 * (100 - df["AttendanceRate"])

    + 0.07 * (100 - df["CompletionRate"])

    + 0.06 * (100 - df["AveragePracticeScore"])

    + 0.05 * (40 - df["StudyHoursPerWeek"])

    + 0.04 * (20 - df["CurrentGPA"])

    + 0.03 * (25 - df["LoginFrequency"])

    - 0.03 * df["Scholarship"]

    + noise

)

probability = 1 / (1 + np.exp(-(risk_score - 8) / 0.35))

df["DropoutRisk"] = (
    np.random.rand(len(df)) < probability
).astype(int)


In [17]:
noise = np.random.normal(0, 0.8, len(df))

performance_score = (

      4.0 * df["CurrentGPA"]

    + 0.35 * df["AssignmentAverage"]

    + 0.30 * df["QuizAverage"]

    + 0.30 * df["MidtermScore"]

    + 0.20 * df["AttendanceRate"]

    + 0.15 * df["AveragePracticeScore"]

    + 0.12 * df["CompletionRate"]

    + 0.08 * df["StudyHoursPerWeek"]

    + 0.05 * df["LoginFrequency"]

    + 0.03 * df["LibraryVisits"]

    - 2.50 * df["PreviousFailures"]

    - 1.50 * df["ConsecutiveInactiveDays"]

    - 1.20 * df["AssignmentSubmissionDelay"]

    + noise

)

q_edges = performance_score.quantile([0, 0.25, 0.5, 0.75, 1.0]).to_numpy().copy()
q_edges[0] = -np.inf
q_edges[-1] = np.inf

df["PerformanceLevel"] = pd.cut(
    performance_score,
    bins=q_edges,
    labels=[
        "Weak",
        "Average",
        "Good",
        "Excellent"
    ],
    include_lowest=True
)


In [18]:
df["PerformanceLevel"] = df["PerformanceLevel"].map({
    "Weak": 0,
    "Average": 1,
    "Good": 2,
    "Excellent": 3
}).astype(int)

### 4. Train-Test Split & Feature Standardization


In [19]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    root_mean_squared_error,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import tensorflow as tf 
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization

In [20]:
X = df.drop(
    columns=[
        "FutureGPA",
        "DropoutRisk",
        "PerformanceLevel"
    ]
)

y_future_gpa = df["FutureGPA"]

y_dropout = df["DropoutRisk"]

y_performance = df["PerformanceLevel"]


In [21]:
(
    X_train,
    X_test,

    y_future_gpa_train,
    y_future_gpa_test,

    y_dropout_train,
    y_dropout_test,

    y_performance_train,
    y_performance_test

) = train_test_split(
    X,
    y_future_gpa,
    y_dropout,
    y_performance,

    test_size=0.2,
    random_state=42
)

In [22]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)

X_test = scaler.transform(X_test)

y_scaler = StandardScaler()
y_future_gpa_train_scaled = y_scaler.fit_transform(
    y_future_gpa_train.values.reshape(-1, 1)
).ravel()


### 5. Multi-Output Deep Neural Network Architecture


In [23]:
input_layer = Input(shape=(X_train.shape[1],))

dense_1 = Dense(256, activation="relu")(input_layer)
batch_norm_1 = BatchNormalization()(dense_1)

dropout_1 = Dropout(0.30, name="dropout_reg_1")(batch_norm_1)

dense_2 = Dense(128, activation="relu")(dropout_1)
batch_norm_2 = BatchNormalization()(dense_2)
dropout_2 = Dropout(0.25, name="dropout_reg_2")(batch_norm_2)

dense_3 = Dense(64, activation="relu")(dropout_2)
batch_norm_3 = BatchNormalization()(dense_3)
dropout_3 = Dropout(0.20, name="dropout_reg_3")(batch_norm_3)

shared_output = Dense(32, activation="relu")(dropout_3)

In [24]:
gpa_branch = Dense(32, activation="relu")(shared_output)
gpa_branch = Dense(16, activation="relu")(gpa_branch)

future_gpa_output = Dense(
    1,
    activation="linear",
    name="future_gpa"
)(gpa_branch)


In [25]:
dropout_branch = Dense(32, activation="relu")(shared_output)
dropout_branch = Dense(16, activation="relu")(dropout_branch)

dropout_output = Dense(
    1,
    activation="sigmoid",
    name="dropout"
)(dropout_branch)

In [26]:
performance_branch = Dense(32, activation="relu")(shared_output)
performance_branch = Dense(16, activation="relu")(performance_branch)

performance_output = Dense(
    4,
    activation="softmax",
    name="performance"
)(performance_branch)

In [27]:
model = Model(
    inputs=input_layer,
    outputs=[
        future_gpa_output,
        dropout_output,
        performance_output
    ]
)

### 6. Model Compilation & Multi-Objective Training


In [28]:
model.compile(

    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss={

        "future_gpa":"mse",
        "dropout":"binary_crossentropy",
        "performance":"sparse_categorical_crossentropy",
    },

    loss_weights={
        "future_gpa": 1.0,
        "dropout": 1.0,
        "performance": 1.5
    },

    metrics={

        "future_gpa":["mae"],
        "dropout":["accuracy"],
        "performance":["accuracy"]
    }

)

In [29]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 39)                │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense (Dense)                 │ (None, 256)               │          10,240 │ input_layer[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization           │ (None, 256)               │           1,024 │ dense[0][0]                │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout_reg_1 (Dropout)       │ (None, 256)               │               0 │ batch_normalization[0][0]  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_1 (Dense)               │ (None, 128)               │          32,896 │ dropout_reg_1[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_1         │ (None, 128)               │             512 │ dense_1[0][0]              │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout_reg_2 (Dropout)       │ (None, 128)               │               0 │ batch_normalization_1[0][… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_2 (Dense)               │ (None, 64)                │           8,256 │ dropout_reg_2[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_2         │ (None, 64)                │             256 │ dense_2[0][0]              │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout_reg_3 (Dropout)       │ (None, 64)                │               0 │ batch_normalization_2[0][… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_3 (Dense)               │ (None, 32)                │           2,080 │ dropout_reg_3[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_4 (Dense)               │ (None, 32)                │           1,056 │ dense_3[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_6 (Dense)               │ (None, 32)                │           1,056 │ dense_3[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_8 (Dense)               │ (None, 32)                │           1,056 │ dense_3[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_5 (Dense)               │ (None, 16)                │             528 │ dense_4[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_7 (Dense)               │ (None, 16)                │             52

 Total params: 60,118 (234.84 KB)

 Trainable params: 59,222 (231.34 KB)

 Non-trainable params: 896 (3.50 KB)

In [30]:
early_stop = EarlyStopping(

    monitor="val_loss",
    patience=25,
    restore_best_weights=True

)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=8,
    min_lr=1e-6
)

In [31]:
history = model.fit(

    X_train,
    {
        "future_gpa": y_future_gpa_train_scaled,
        "dropout": y_dropout_train,
        "performance": y_performance_train
    },

    validation_split=0.2,
    epochs=300,
    batch_size=32,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

Epoch 1/300
100/100 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - dropout_accuracy: 0.7106 - dropout_loss: 0.5547 - future_gpa_loss: 0.5218 - future_gpa_mae: 0.5327 - loss: 2.7270 - performance_accuracy: 0.4756 - performance_loss: 1.1004 - val_dropout_accuracy: 0.8500 - val_dropout_loss: 0.4116 - val_future_gpa_loss: 0.2627 - val_future_gpa_mae: 0.4175 - val_loss: 1.8853 - val_performance_accuracy: 0.6787 - val_performance_loss: 0.8073 - learning_rate: 0.0010
Epoch 2/300
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - dropout_accuracy: 0.8591 - dropout_loss: 0.3366 - future_gpa_loss: 0.1575 - future_gpa_mae: 0.3130 - loss: 1.4617 - performance_accuracy: 0.7188 - performance_loss: 0.6450 - val_dropout_accuracy: 0.9062 - val_dropout_loss: 0.2440 - val_future_gpa_loss: 0.0739 - val_future_gpa_mae: 0.2187 - val_loss: 0.9574 - val_performance_accuracy: 0.8375 - val_performance_loss: 0.4263 - learning_rate: 0.0010
Epoch 3/300
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - dropout_accuracy: 0.8753 - dropout_loss

### 7. Comprehensive Model Evaluation Across All 3 Tasks


In [32]:
results = model.evaluate(

    X_test,

    {
        "future_gpa": y_scaler.transform(y_future_gpa_test.values.reshape(-1, 1)).ravel(),
        "dropout": y_dropout_test,
        "performance": y_performance_test
    },

    verbose=0

)

future_gpa_pred_scaled, dropout_pred, performance_pred = model.predict(X_test)


future_gpa_pred = y_scaler.inverse_transform(
    future_gpa_pred_scaled.reshape(-1, 1)
).ravel()

32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step  


In [33]:
mae = mean_absolute_error(
    y_future_gpa_test,
    future_gpa_pred
)

rmse = np.sqrt(
    mean_squared_error(
        y_future_gpa_test,
        future_gpa_pred
    )
)

r2 = r2_score(
    y_future_gpa_test,
    future_gpa_pred
)

print("Future Gpa\n")

print("MAE :", mae)

print("RMSE:", rmse)

print("R²  :", r2)

Future Gpa

MAE : 0.24988041007995604
RMSE: 0.3156371126216857
R²  : 0.9167336454384595


In [34]:
dropout_pred_class = (
    dropout_pred > 0.5
).astype(int)

dropout_accuracy = accuracy_score(
    y_dropout_test,
    dropout_pred_class
)

dropout_precision = precision_score(
    y_dropout_test,
    dropout_pred_class
)

dropout_recall = recall_score(
    y_dropout_test,
    dropout_pred_class
)

dropout_f1 = f1_score(
    y_dropout_test,
    dropout_pred_class
)

print("Dropout\n")

print("accuracy :", dropout_accuracy)

print("precision:", dropout_precision)

print("recall  :", dropout_recall)

print("f1 score:",dropout_f1)

Dropout

accuracy : 0.918
precision: 0.9321428571428572
recall  : 0.9222614840989399
f1 score: 0.9271758436944938


In [35]:
performance_pred_class = np.argmax(
    performance_pred,
    axis=1
)

performance_accuracy = accuracy_score(
    y_performance_test,
    performance_pred_class
)

performance_precision = precision_score(
    y_performance_test,
    performance_pred_class,
    average="weighted"
)

performance_recall = recall_score(
    y_performance_test,
    performance_pred_class,
    average="weighted"
)

performance_f1 = f1_score(
    y_performance_test,
    performance_pred_class,
    average="weighted"
)

print("performance\n")

print("accuracy :", performance_accuracy)

print("precision:", performance_precision)

print("recall  :", performance_recall)

print("f1 score:",performance_f1)

performance

accuracy : 0.915
precision: 0.9163428932178932
recall  : 0.915
f1 score: 0.9153481106558493


### 8. Exporting Production Artifacts for Streamlit App


In [36]:
# ==========================================
# Save everything the Streamlit app needs
# ==========================================
import os
import json
import joblib
 
os.makedirs("artifacts", exist_ok=True)
 
model.save("artifacts/model.keras")
joblib.dump(scaler, "artifacts/scaler.pkl")
joblib.dump(y_scaler, "artifacts/y_scaler.pkl")
 
feature_columns = X.columns.tolist()
with open("artifacts/feature_columns.json", "w") as f:
    json.dump(feature_columns, f, ensure_ascii=False, indent=2)
 
# Save min/mean/max per feature so the Streamlit UI can build sensible
# sliders/defaults without the app needing to regenerate the dataset.
feature_stats = {}
for col in feature_columns:
    feature_stats[col] = {
        "min": float(X[col].min()),
        "max": float(X[col].max()),
        "mean": float(X[col].mean()),
        "dtype": "int" if str(X[col].dtype).startswith("int") else "float"
    }
with open("artifacts/feature_stats.json", "w") as f:
    json.dump(feature_stats, f, ensure_ascii=False, indent=2)
 
test_metrics = {
    "future_gpa": {"mae": float(mae), "rmse": float(rmse), "r2": float(r2)},
    "dropout": {
        "accuracy": float(dropout_accuracy),
        "precision": float(dropout_precision),
        "recall": float(dropout_recall),
        "f1": float(dropout_f1),
    },
    "performance": {
        "accuracy": float(performance_accuracy),
        "precision": float(performance_precision),
        "recall": float(performance_recall),
        "f1": float(performance_f1),
    },
}
with open("artifacts/test_metrics.json", "w") as f:
    json.dump(test_metrics, f, ensure_ascii=False, indent=2)
 
print("\nSaved model + scalers + feature metadata to ./artifacts/")


Saved model + scalers + feature metadata to ./artifacts/


In [37]:
import json

import joblib
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import streamlit as st
from tensorflow.keras.models import load_model

# ==========================================================
# Page config + RTL styling (Persian-friendly)
# ==========================================================
st.set_page_config(
    page_title="پیش‌بینی وضعیت تحصیلی دانشجو",
    page_icon="🎓",
    layout="wide",
)

st.markdown(
    """
    <style>
    html, body, [class*="css"]  {
        direction: rtl;
        text-align: right;
        font-family: 'Vazirmatn', 'Tahoma', sans-serif;
    }
    [data-testid="stMetricLabel"] { direction: rtl; }
    [data-testid="stSlider"] label { direction: rtl; }
    .stSlider > div { direction: ltr; }
    div[data-baseweb="select"] { direction: rtl; }
    </style>
    """,
    unsafe_allow_html=True,
)

ARTIFACTS_DIR = "artifacts"


# ==========================================================
# Load model + preprocessing artifacts (cached so it only
# happens once per server session, not on every interaction)
# ==========================================================
@st.cache_resource
def load_artifacts():
    model = load_model(f"{ARTIFACTS_DIR}/model.keras")
    scaler = joblib.load(f"{ARTIFACTS_DIR}/scaler.pkl")
    y_scaler = joblib.load(f"{ARTIFACTS_DIR}/y_scaler.pkl")
    with open(f"{ARTIFACTS_DIR}/feature_columns.json", encoding="utf-8") as f:
        feature_columns = json.load(f)
    with open(f"{ARTIFACTS_DIR}/feature_stats.json", encoding="utf-8") as f:
        feature_stats = json.load(f)
    with open(f"{ARTIFACTS_DIR}/test_metrics.json", encoding="utf-8") as f:
        test_metrics = json.load(f)
    return model, scaler, y_scaler, feature_columns, feature_stats, test_metrics


try:
    model, scaler, y_scaler, feature_columns, feature_stats, test_metrics = load_artifacts()
except Exception as e:
    st.error(
        "فایل‌های مدل پیدا نشدن. اول باید اسکریپت `train_and_save.py` رو یک بار اجرا کنید "
        "تا پوشه‌ی `artifacts/` (شامل model.keras, scaler.pkl, ...) ساخته بشه.\n\n"
        f"جزئیات خطا: {e}"
    )
    st.stop()


def slider_for(col, label, section_key, help_text=None, step=None):
    """Build a slider for a continuous/discrete numeric feature using the
    saved min/max/mean stats, keeping the value in Streamlit session_state
    so it survives reruns."""
    stats = feature_stats[col]
    is_int = stats["dtype"] == "int"
    min_v = stats["min"]
    max_v = stats["max"]
    mean_v = stats["mean"]

    if is_int:
        min_v, max_v, mean_v = int(min_v), int(max_v), int(round(mean_v))
        step = step or 1
    else:
        step = step or round((max_v - min_v) / 100, 2) or 0.1
        min_v, max_v, mean_v = float(min_v), float(max_v), round(float(mean_v), 2)

    return st.slider(
        label,
        min_value=min_v,
        max_value=max_v,
        value=mean_v,
        step=step,
        help=help_text,
        key=f"{section_key}_{col}",
    )


# ==========================================================
# Header
# ==========================================================
st.title("🎓 سامانه پیش‌بینی وضعیت تحصیلی دانشجو")
st.caption(
    "با وارد کردن مشخصات یک دانشجو، مدل شبکه‌ی عصبی سه خروجی رو هم‌زمان پیش‌بینی می‌کنه: "
    "معدل ترم آینده، ریسک انصراف از تحصیل، و سطح عملکرد کلی."
)

with st.expander("📊 دقت مدل روی داده‌ی آزمایشی (Test Set)"):
    c1, c2, c3 = st.columns(3)
    c1.metric("R² معدل آینده", f"{test_metrics['future_gpa']['r2']:.1%}")
    c2.metric("دقت ریسک انصراف", f"{test_metrics['dropout']['accuracy']:.1%}")
    c3.metric("دقت سطح عملکرد", f"{test_metrics['performance']['accuracy']:.1%}")

st.divider()

# ==========================================================
# Input form, organized in tabs
# ==========================================================
tab1, tab2, tab3 = st.tabs(
    ["👤 اطلاعات شخصی و خانوادگی", "📚 عملکرد تحصیلی", "💻 فعالیت آنلاین"]
)

values = {}

with tab1:
    col1, col2 = st.columns(2)
    with col1:
        values["Age"] = slider_for("Age", "سن", "p1")
        values["Gender"] = st.selectbox(
            "جنسیت (کدگذاری شده در دیتاست اصلی)", [0, 1], index=0, key="p1_gender"
        )
        values["Department"] = st.selectbox(
            "دپارتمان",
            [0, 1, 2, 3, 4],
            format_func=lambda x: f"دپارتمان {x + 1}",
            key="p1_dept",
        )
        values["Semester"] = slider_for("Semester", "ترم تحصیلی", "p1")
    with col2:
        values["FamilyIncome"] = slider_for(
            "FamilyIncome", "درآمد خانواده (میلیون تومان)", "p1"
        )
        values["ParentEducation"] = st.selectbox(
            "تحصیلات والدین",
            [0, 1, 2, 3],
            format_func=lambda x: ["دیپلم", "کاردانی", "کارشناسی", "ارشد/دکترا"][x],
            index=1,
            key="p1_pedu",
        )
        values["Scholarship"] = st.selectbox(
            "بورسیه‌ی تحصیلی دارد؟",
            [0, 1],
            format_func=lambda x: "خیر" if x == 0 else "بله",
            key="p1_schol",
        )

with tab2:
    col1, col2 = st.columns(2)
    with col1:
        values["HighSchoolGPA"] = slider_for("HighSchoolGPA", "معدل دبیرستان (از ۲۰)", "p2")
        values["CurrentGPA"] = slider_for("CurrentGPA", "معدل فعلی (از ۲۰)", "p2")
        values["StudyHoursPerWeek"] = slider_for(
            "StudyHoursPerWeek", "ساعت مطالعه در هفته", "p2"
        )
        values["SleepHours"] = slider_for("SleepHours", "میانگین ساعت خواب روزانه", "p2")
        values["AttendanceRate"] = slider_for("AttendanceRate", "نرخ حضور در کلاس (٪)", "p2")
        values["LibraryVisits"] = slider_for("LibraryVisits", "تعداد مراجعه به کتابخانه", "p2")
    with col2:
        values["AssignmentAverage"] = slider_for(
            "AssignmentAverage", "میانگین نمره‌ی تکالیف", "p2"
        )
        values["QuizAverage"] = slider_for("QuizAverage", "میانگین نمره‌ی کوییزها", "p2")
        values["MidtermScore"] = slider_for("MidtermScore", "نمره‌ی میان‌ترم", "p2")
        values["ClassParticipation"] = slider_for(
            "ClassParticipation", "مشارکت کلاسی", "p2"
        )
        values["PreviousFailures"] = slider_for(
            "PreviousFailures", "تعداد افت تحصیلی‌های قبلی", "p2"
        )
        values["InternetUsage"] = slider_for(
            "InternetUsage", "ساعت استفاده‌ی روزانه از اینترنت (غیردرسی)", "p2"
        )

with tab3:
    col1, col2 = st.columns(2)
    with col1:
        values["LoginFrequency"] = slider_for(
            "LoginFrequency", "تعداد ورود به سامانه در هفته", "p3"
        )
        values["TotalLoginTime"] = slider_for(
            "TotalLoginTime", "کل زمان حضور آنلاین (دقیقه/هفته)", "p3"
        )
        values["VideosWatched"] = slider_for("VideosWatched", "تعداد ویدیوهای دیده‌شده", "p3")
        values["CompletionRate"] = slider_for(
            "CompletionRate", "نرخ تکمیل دوره (٪)", "p3"
        )
        values["AveragePracticeScore"] = slider_for(
            "AveragePracticeScore", "میانگین نمره‌ی تمرین‌های آنلاین", "p3"
        )
        values["PracticeTestsTaken"] = slider_for(
            "PracticeTestsTaken", "تعداد آزمون‌های تمرینی", "p3"
        )
        values["DownloadedResources"] = slider_for(
            "DownloadedResources", "تعداد منابع دانلودشده", "p3"
        )
        values["AssignmentSubmissionDelay"] = slider_for(
            "AssignmentSubmissionDelay", "میانگین تأخیر در ارسال تکلیف (روز)", "p3"
        )
        values["ConsecutiveInactiveDays"] = slider_for(
            "ConsecutiveInactiveDays", "روزهای متوالی غیرفعال بودن", "p3"
        )
    with col2:
        values["ForumPosts"] = slider_for("ForumPosts", "تعداد پست‌های فروم", "p3")
        values["QuestionsAsked"] = slider_for("QuestionsAsked", "تعداد سؤالات پرسیده‌شده", "p3")
        values["QuestionsAnswered"] = slider_for(
            "QuestionsAnswered", "تعداد سؤالات پاسخ‌داده‌شده", "p3"
        )
        values["MobileUsage"] = slider_for("MobileUsage", "درصد استفاده از موبایل (٪)", "p3")
        values["DesktopUsage"] = slider_for("DesktopUsage", "درصد استفاده از دسکتاپ (٪)", "p3")
        values["NightStudyPercentage"] = slider_for(
            "NightStudyPercentage", "درصد مطالعه‌ی شبانه (٪)", "p3"
        )
        values["WeekendStudyPercentage"] = slider_for(
            "WeekendStudyPercentage", "درصد مطالعه در آخر هفته (٪)", "p3"
        )
        values["AIUsageFrequency"] = slider_for(
            "AIUsageFrequency", "دفعات استفاده از ابزارهای هوش مصنوعی", "p3"
        )
        values["CourseCompletionTime"] = slider_for(
            "CourseCompletionTime", "زمان تخمینی تکمیل دوره (هفته)", "p3"
        )
        values["CertificateEarned"] = st.selectbox(
            "مدرک/گواهی دوره را کسب کرده؟",
            [0, 1],
            format_func=lambda x: "خیر" if x == 0 else "بله",
            key="p3_cert",
        )

st.divider()
predict_clicked = st.button("🔮 پیش‌بینی کن", type="primary", use_container_width=True)

# ==========================================================
# Prediction
# ==========================================================
if predict_clicked:
    # StudentID carries no real signal (it's just a sequential row id in the
    # training data) -- fill it with the training-set mean rather than
    # exposing a meaningless slider to the user.
    values["StudentID"] = feature_stats["StudentID"]["mean"]

    row = pd.DataFrame([[values[c] for c in feature_columns]], columns=feature_columns)
    X_scaled = scaler.transform(row)

    future_gpa_pred_scaled, dropout_pred, performance_pred = model.predict(
        X_scaled, verbose=0
    )
    future_gpa = float(
        y_scaler.inverse_transform(future_gpa_pred_scaled.reshape(-1, 1)).ravel()[0]
    )
    future_gpa = float(np.clip(future_gpa, 0, 20))

    dropout_prob = float(dropout_pred.ravel()[0])
    perf_probs = performance_pred.ravel()
    perf_labels = ["ضعیف", "متوسط", "خوب", "عالی"]
    perf_idx = int(np.argmax(perf_probs))

    st.subheader("نتیجه‌ی پیش‌بینی")
    r1, r2, r3 = st.columns(3)

    # ---- FutureGPA gauge ----
    with r1:
        fig = go.Figure(
            go.Indicator(
                mode="gauge+number",
                value=future_gpa,
                number={"suffix": " / 20"},
                title={"text": "معدل پیش‌بینی‌شده‌ی ترم آینده"},
                gauge={
                    "axis": {"range": [0, 20]},
                    "bar": {"color": "#2563eb"},
                    "steps": [
                        {"range": [0, 10], "color": "#fee2e2"},
                        {"range": [10, 14], "color": "#fef9c3"},
                        {"range": [14, 20], "color": "#dcfce7"},
                    ],
                },
            )
        )
        fig.update_layout(height=280, margin=dict(l=20, r=20, t=50, b=10))
        st.plotly_chart(fig, use_container_width=True)

    # ---- Dropout risk ----
    with r2:
        risk_color = "#dc2626" if dropout_prob >= 0.5 else "#16a34a"
        risk_label = "پرخطر ⚠️" if dropout_prob >= 0.5 else "کم‌خطر ✅"
        fig = go.Figure(
            go.Indicator(
                mode="gauge+number",
                value=dropout_prob * 100,
                number={"suffix": "%"},
                title={"text": f"ریسک انصراف از تحصیل — {risk_label}"},
                gauge={
                    "axis": {"range": [0, 100]},
                    "bar": {"color": risk_color},
                    "steps": [
                        {"range": [0, 50], "color": "#dcfce7"},
                        {"range": [50, 100], "color": "#fee2e2"},
                    ],
                },
            )
        )
        fig.update_layout(height=280, margin=dict(l=20, r=20, t=50, b=10))
        st.plotly_chart(fig, use_container_width=True)

    # ---- Performance level ----
    with r3:
        fig = go.Figure(
            go.Bar(
                x=perf_probs * 100,
                y=perf_labels,
                orientation="h",
                marker_color=["#dc2626", "#f97316", "#3b82f6", "#16a34a"],
                text=[f"{p:.0%}" for p in perf_probs],
                textposition="outside",
            )
        )
        fig.update_layout(
            title=f"سطح عملکرد پیش‌بینی‌شده: {perf_labels[perf_idx]}",
            xaxis_title="احتمال (٪)",
            height=280,
            margin=dict(l=20, r=20, t=50, b=10),
            xaxis_range=[0, 100],
        )
        st.plotly_chart(fig, use_container_width=True)

    st.info(
        "این پیش‌بینی‌ها بر پایه‌ی یک دیتاست **مصنوعی** آموزش دیده‌اند و صرفاً برای اهداف "
        "آموزشی/پورتفولیو هستند، نه تصمیم‌گیری واقعی درباره‌ی دانشجویان."
    )
else:
    st.caption("مقادیر رو تنظیم کنید و روی «پیش‌بینی کن» بزنید.")

2026-08-04 23:15:10.407 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-04 23:15:10.408 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-04 23:15:11.774 
  command:

    streamlit run D:\anaconda\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-08-04 23:15:11.774 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-04 23:15:11.775 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-04 23:15:11.776 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-04 23:15:11.776 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare m

### 9. Streamlit Dashboard Execution
The standalone application is encapsulated in pp.py. To run the dashboard:
`ash
streamlit run app.py
`


In [38]:
%%writefile app.py
import json

import joblib
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import streamlit as st
from tensorflow.keras.models import load_model

# ==========================================================
# Page config + RTL styling (Persian-friendly)
# ==========================================================
st.set_page_config(
    page_title="پیش‌بینی وضعیت تحصیلی دانشجو",
    page_icon="🎓",
    layout="wide",
)

st.markdown(
    """
    <style>
    html, body, [class*="css"]  {
        direction: rtl;
        text-align: right;
        font-family: 'Vazirmatn', 'Tahoma', sans-serif;
    }
    [data-testid="stMetricLabel"] { direction: rtl; }
    [data-testid="stSlider"] label { direction: rtl; }
    .stSlider > div { direction: ltr; }
    div[data-baseweb="select"] { direction: rtl; }
    </style>
    """,
    unsafe_allow_html=True,
)

ARTIFACTS_DIR = "artifacts"


@st.cache_resource
def load_artifacts():
    model = load_model(f"{ARTIFACTS_DIR}/model.keras")
    scaler = joblib.load(f"{ARTIFACTS_DIR}/scaler.pkl")
    y_scaler = joblib.load(f"{ARTIFACTS_DIR}/y_scaler.pkl")
    with open(f"{ARTIFACTS_DIR}/feature_columns.json", encoding="utf-8") as f:
        feature_columns = json.load(f)
    with open(f"{ARTIFACTS_DIR}/feature_stats.json", encoding="utf-8") as f:
        feature_stats = json.load(f)
    with open(f"{ARTIFACTS_DIR}/test_metrics.json", encoding="utf-8") as f:
        test_metrics = json.load(f)
    return model, scaler, y_scaler, feature_columns, feature_stats, test_metrics


try:
    model, scaler, y_scaler, feature_columns, feature_stats, test_metrics = load_artifacts()
except Exception as e:
    st.error(
        "فایل‌های مدل پیدا نشدن. اول باید سلول ذخیره‌سازی artifacts رو اجرا کنید.\n\n"
        f"جزئیات خطا: {e}"
    )
    st.stop()


def slider_for(col, label, section_key, help_text=None, step=None):
    stats = feature_stats[col]
    is_int = stats["dtype"] == "int"
    min_v = stats["min"]
    max_v = stats["max"]
    mean_v = stats["mean"]

    if is_int:
        min_v, max_v, mean_v = int(min_v), int(max_v), int(round(mean_v))
        step = step or 1
    else:
        step = step or round((max_v - min_v) / 100, 2) or 0.1
        min_v, max_v, mean_v = float(min_v), float(max_v), round(float(mean_v), 2)

    return st.slider(
        label,
        min_value=min_v,
        max_value=max_v,
        value=mean_v,
        step=step,
        help=help_text,
        key=f"{section_key}_{col}",
    )


st.title("🎓 سامانه پیش‌بینی وضعیت تحصیلی دانشجو")
st.caption(
    "با وارد کردن مشخصات یک دانشجو، مدل شبکه‌ی عصبی سه خروجی رو هم‌زمان پیش‌بینی می‌کنه: "
    "معدل ترم آینده، ریسک انصراف از تحصیل، و سطح عملکرد کلی."
)

with st.expander("📊 دقت مدل روی داده‌ی آزمایشی (Test Set)"):
    c1, c2, c3 = st.columns(3)
    c1.metric("R² معدل آینده", f"{test_metrics['future_gpa']['r2']:.1%}")
    c2.metric("دقت ریسک انصراف", f"{test_metrics['dropout']['accuracy']:.1%}")
    c3.metric("دقت سطح عملکرد", f"{test_metrics['performance']['accuracy']:.1%}")

st.divider()

tab1, tab2, tab3 = st.tabs(
    ["👤 اطلاعات شخصی و خانوادگی", "📚 عملکرد تحصیلی", "💻 فعالیت آنلاین"]
)

values = {}

with tab1:
    col1, col2 = st.columns(2)
    with col1:
        values["Age"] = slider_for("Age", "سن", "p1")
        values["Gender"] = st.selectbox(
            "جنسیت (کدگذاری شده در دیتاست اصلی)", [0, 1], index=0, key="p1_gender"
        )
        values["Department"] = st.selectbox(
            "دپارتمان",
            [0, 1, 2, 3, 4],
            format_func=lambda x: f"دپارتمان {x + 1}",
            key="p1_dept",
        )
        values["Semester"] = slider_for("Semester", "ترم تحصیلی", "p1")
    with col2:
        values["FamilyIncome"] = slider_for(
            "FamilyIncome", "درآمد خانواده (میلیون تومان)", "p1"
        )
        values["ParentEducation"] = st.selectbox(
            "تحصیلات والدین",
            [0, 1, 2, 3],
            format_func=lambda x: ["دیپلم", "کاردانی", "کارشناسی", "ارشد/دکترا"][x],
            index=1,
            key="p1_pedu",
        )
        values["Scholarship"] = st.selectbox(
            "بورسیه‌ی تحصیلی دارد؟",
            [0, 1],
            format_func=lambda x: "خیر" if x == 0 else "بله",
            key="p1_schol",
        )

with tab2:
    col1, col2 = st.columns(2)
    with col1:
        values["HighSchoolGPA"] = slider_for("HighSchoolGPA", "معدل دبیرستان (از ۲۰)", "p2")
        values["CurrentGPA"] = slider_for("CurrentGPA", "معدل فعلی (از ۲۰)", "p2")
        values["StudyHoursPerWeek"] = slider_for(
            "StudyHoursPerWeek", "ساعت مطالعه در هفته", "p2"
        )
        values["SleepHours"] = slider_for("SleepHours", "میانگین ساعت خواب روزانه", "p2")
        values["AttendanceRate"] = slider_for("AttendanceRate", "نرخ حضور در کلاس (٪)", "p2")
        values["LibraryVisits"] = slider_for("LibraryVisits", "تعداد مراجعه به کتابخانه", "p2")
    with col2:
        values["AssignmentAverage"] = slider_for(
            "AssignmentAverage", "میانگین نمره‌ی تکالیف", "p2"
        )
        values["QuizAverage"] = slider_for("QuizAverage", "میانگین نمره‌ی کوییزها", "p2")
        values["MidtermScore"] = slider_for("MidtermScore", "نمره‌ی میان‌ترم", "p2")
        values["ClassParticipation"] = slider_for(
            "ClassParticipation", "مشارکت کلاسی", "p2"
        )
        values["PreviousFailures"] = slider_for(
            "PreviousFailures", "تعداد افت تحصیلی‌های قبلی", "p2"
        )
        values["InternetUsage"] = slider_for(
            "InternetUsage", "ساعت استفاده‌ی روزانه از اینترنت (غیردرسی)", "p2"
        )

with tab3:
    col1, col2 = st.columns(2)
    with col1:
        values["LoginFrequency"] = slider_for(
            "LoginFrequency", "تعداد ورود به سامانه در هفته", "p3"
        )
        values["TotalLoginTime"] = slider_for(
            "TotalLoginTime", "کل زمان حضور آنلاین (دقیقه/هفته)", "p3"
        )
        values["VideosWatched"] = slider_for("VideosWatched", "تعداد ویدیوهای دیده‌شده", "p3")
        values["CompletionRate"] = slider_for(
            "CompletionRate", "نرخ تکمیل دوره (٪)", "p3"
        )
        values["AveragePracticeScore"] = slider_for(
            "AveragePracticeScore", "میانگین نمره‌ی تمرین‌های آنلاین", "p3"
        )
        values["PracticeTestsTaken"] = slider_for(
            "PracticeTestsTaken", "تعداد آزمون‌های تمرینی", "p3"
        )
        values["DownloadedResources"] = slider_for(
            "DownloadedResources", "تعداد منابع دانلودشده", "p3"
        )
        values["AssignmentSubmissionDelay"] = slider_for(
            "AssignmentSubmissionDelay", "میانگین تأخیر در ارسال تکلیف (روز)", "p3"
        )
        values["ConsecutiveInactiveDays"] = slider_for(
            "ConsecutiveInactiveDays", "روزهای متوالی غیرفعال بودن", "p3"
        )
    with col2:
        values["ForumPosts"] = slider_for("ForumPosts", "تعداد پست‌های فروم", "p3")
        values["QuestionsAsked"] = slider_for("QuestionsAsked", "تعداد سؤالات پرسیده‌شده", "p3")
        values["QuestionsAnswered"] = slider_for(
            "QuestionsAnswered", "تعداد سؤالات پاسخ‌داده‌شده", "p3"
        )
        values["MobileUsage"] = slider_for("MobileUsage", "درصد استفاده از موبایل (٪)", "p3")
        values["DesktopUsage"] = slider_for("DesktopUsage", "درصد استفاده از دسکتاپ (٪)", "p3")
        values["NightStudyPercentage"] = slider_for(
            "NightStudyPercentage", "درصد مطالعه‌ی شبانه (٪)", "p3"
        )
        values["WeekendStudyPercentage"] = slider_for(
            "WeekendStudyPercentage", "درصد مطالعه در آخر هفته (٪)", "p3"
        )
        values["AIUsageFrequency"] = slider_for(
            "AIUsageFrequency", "دفعات استفاده از ابزارهای هوش مصنوعی", "p3"
        )
        values["CourseCompletionTime"] = slider_for(
            "CourseCompletionTime", "زمان تخمینی تکمیل دوره (هفته)", "p3"
        )
        values["CertificateEarned"] = st.selectbox(
            "مدرک/گواهی دوره را کسب کرده؟",
            [0, 1],
            format_func=lambda x: "خیر" if x == 0 else "بله",
            key="p3_cert",
        )

st.divider()
predict_clicked = st.button("🔮 پیش‌بینی کن", type="primary", use_container_width=True)

if predict_clicked:
    values["StudentID"] = feature_stats["StudentID"]["mean"]

    row = pd.DataFrame([[values[c] for c in feature_columns]], columns=feature_columns)
    X_scaled = scaler.transform(row)

    future_gpa_pred_scaled, dropout_pred, performance_pred = model.predict(
        X_scaled, verbose=0
    )
    future_gpa = float(
        y_scaler.inverse_transform(future_gpa_pred_scaled.reshape(-1, 1)).ravel()[0]
    )
    future_gpa = float(np.clip(future_gpa, 0, 20))

    dropout_prob = float(dropout_pred.ravel()[0])
    perf_probs = performance_pred.ravel()
    perf_labels = ["ضعیف", "متوسط", "خوب", "عالی"]
    perf_idx = int(np.argmax(perf_probs))

    st.subheader("نتیجه‌ی پیش‌بینی")
    r1, r2, r3 = st.columns(3)

    with r1:
        fig = go.Figure(
            go.Indicator(
                mode="gauge+number",
                value=future_gpa,
                number={"suffix": " / 20"},
                title={"text": "معدل پیش‌بینی‌شده‌ی ترم آینده"},
                gauge={
                    "axis": {"range": [0, 20]},
                    "bar": {"color": "#2563eb"},
                    "steps": [
                        {"range": [0, 10], "color": "#fee2e2"},
                        {"range": [10, 14], "color": "#fef9c3"},
                        {"range": [14, 20], "color": "#dcfce7"},
                    ],
                },
            )
        )
        fig.update_layout(height=280, margin=dict(l=20, r=20, t=50, b=10))
        st.plotly_chart(fig, use_container_width=True)

    with r2:
        risk_color = "#dc2626" if dropout_prob >= 0.5 else "#16a34a"
        risk_label = "پرخطر ⚠️" if dropout_prob >= 0.5 else "کم‌خطر ✅"
        fig = go.Figure(
            go.Indicator(
                mode="gauge+number",
                value=dropout_prob * 100,
                number={"suffix": "%"},
                title={"text": f"ریسک انصراف از تحصیل — {risk_label}"},
                gauge={
                    "axis": {"range": [0, 100]},
                    "bar": {"color": risk_color},
                    "steps": [
                        {"range": [0, 50], "color": "#dcfce7"},
                        {"range": [50, 100], "color": "#fee2e2"},
                    ],
                },
            )
        )
        fig.update_layout(height=280, margin=dict(l=20, r=20, t=50, b=10))
        st.plotly_chart(fig, use_container_width=True)

    with r3:
        fig = go.Figure(
            go.Bar(
                x=perf_probs * 100,
                y=perf_labels,
                orientation="h",
                marker_color=["#dc2626", "#f97316", "#3b82f6", "#16a34a"],
                text=[f"{p:.0%}" for p in perf_probs],
                textposition="outside",
            )
        )
        fig.update_layout(
            title=f"سطح عملکرد پیش‌بینی‌شده: {perf_labels[perf_idx]}",
            xaxis_title="احتمال (٪)",
            height=280,
            margin=dict(l=20, r=20, t=50, b=10),
            xaxis_range=[0, 100],
        )
        st.plotly_chart(fig, use_container_width=True)

    st.info(
        "این پیش‌بینی‌ها بر پایه‌ی یک دیتاست **مصنوعی** آموزش دیده‌اند و صرفاً برای اهداف "
        "آموزشی/پورتفولیو هستند، نه تصمیم‌گیری واقعی درباره‌ی دانشجویان."
    )
else:
    st.caption("مقادیر رو تنظیم کنید و روی «پیش‌بینی کن» بزنید.")

Writing app.py


In [39]:
import subprocess
import time
import webbrowser

PORT = 8501

# اگه نسخه‌ی قبلی روی همین پورت در حال اجراست، اول متوقفش می‌کنیم
try:
    streamlit_process.terminate()
    time.sleep(1)
except NameError:
    pass

streamlit_process = subprocess.Popen(
    ["streamlit", "run", "app.py",
     "--server.port", str(PORT),
     "--server.headless", "true"]
)

time.sleep(6)  # فرصت برای بالا اومدن سرور

url = f"http://localhost:{PORT}"
print(f"✅ اپلیکیشن روی این آدرس در حال اجراست: {url}")
webbrowser.open(url)

✅ اپلیکیشن روی این آدرس در حال اجراست: http://localhost:8501


True

In [ ]:
streamlit_process.terminate()